# Data Exploration
====================

Explore raw data from PostgreSQL, Kaggle, and MinIO for manual inspection and testing.

In [ ]:
# Connect to services and load data
import os
import sys
import pandas as pd
import boto3
from datetime import datetime

sys.path.insert(0, '..')

# Configuration
MINIO_CONFIG = {
    "endpoint": os.getenv("MINIO_ENDPOINT", "localhost:9900"),
    "access_key": os.getenv("MINIO_ROOT_USER", "minioadmin"),
    "secret_key": os.getenv("MINIO_ROOT_PASSWORD", "minioadmin"),
    "bucket": os.getenv("MINIO_BUCKET", "insurance-data"),
}

POSTGRES_CONFIG = {
    "host": os.getenv("POSTGRES_HOST", "localhost"),
    "port": os.getenv("POSTGRES_PORT", "5435"),
    "db": os.getenv("POSTGRES_DB", "insurance_db"),
    "user": os.getenv("POSTGRES_USER", "insurance_user"),
    "password": os.getenv("POSTGRES_PASSWORD", "insurance_pass"),
}

def get_minio_client():
    return boto3.client(
        "s3",
        endpoint_url=f"http://{MINIO_CONFIG['endpoint']}",
        aws_access_key_id=MINIO_CONFIG["access_key"],
        aws_secret_access_key=MINIO_CONFIG["secret_key"],
    )

s3 = get_minio_client()
print("MinIO client connected")

## List Raw Objects in MinIO

In [ ]:
# List all objects in raw/ prefix
def list_raw_objects(prefix: str = "raw/"):
    response = s3.list_objects_v2(
        Bucket=MINIO_CONFIG["bucket"],
        Prefix=prefix,
    )
    if "Contents" not in response:
        return []
    return [obj["Key"] for obj in response["Contents"]]

print("=== Raw Data ===")
for key in list_raw_objects("raw/"):
    print(f"  {key}")

print("\n=== Silver Data ===")
for key in list_raw_objects("silver/"):
    print(f"  {key}")

## Load Raw Customers

In [ ]:
# Load PostgreSQL customers from raw/customers/
def read_parquet_from_s3(key: str) -> pd.DataFrame:
    local_path = f"/tmp/{os.path.basename(key)}"
    s3.download_file(MINIO_CONFIG["bucket"], key, local_path)
    df = pd.read_parquet(local_path)
    os.unlink(local_path)
    return df

pg_customer_keys = [k for k in list_raw_objects("raw/customers/") if k.endswith(".parquet")]
print(f"Found {len(pg_customer_keys)} customer files")

if pg_customer_keys:
    df_customers = read_parquet_from_s3(pg_customer_keys[0])
    print(f"\nRaw Customers: {len(df_customers)} rows")
    print(df_customers.head())
    print("\nColumns:", list(df_customers.columns))
    print("\nData Types:")
    print(df_customers.dtypes)

## Load Raw Claims

In [ ]:
# Load PostgreSQL claims from raw/claims/
pg_claim_keys = [k for k in list_raw_objects("raw/claims/") if k.endswith(".parquet")]
print(f"Found {len(pg_claim_keys)} claim files")

if pg_claim_keys:
    df_claims = read_parquet_from_s3(pg_claim_keys[0])
    print(f"\nRaw Claims: {len(df_claims)} rows")
    print(df_claims.head())
    print("\nColumns:", list(df_claims.columns))
    print("\nData Types:")
    print(df_claims.dtypes)

## Load Kaggle Data

In [ ]:
# Load Kaggle customers
kg_customer_keys = [k for k in list_raw_objects("raw/kaggle_customers/") if k.endswith(".parquet")]
print(f"Found {len(kg_customer_keys)} Kaggle customer files")

if kg_customer_keys:
    df_kg_customers = read_parquet_from_s3(kg_customer_keys[0])
    print(f"\nKaggle Customers: {len(df_kg_customers)} rows")
    print(df_kg_customers.head())
    print("\nColumns:", list(df_kg_customers.columns))

# Load Kaggle claims
kg_claim_keys = [k for k in list_raw_objects("raw/kaggle_claims/") if k.endswith(".parquet")]
print(f"\nFound {len(kg_claim_keys)} Kaggle claim files")

if kg_claim_keys:
    df_kg_claims = read_parquet_from_s3(kg_claim_keys[0])
    print(f"\nKaggle Claims: {len(df_kg_claims)} rows")
    print(df_kg_claims.head())
    print("\nColumns:", list(df_kg_claims.columns))

## Query PostgreSQL Directly

In [ ]:
import psycopg2

# Connect to PostgreSQL
conn = psycopg2.connect(
    host=POSTGRES_CONFIG["host"],
    port=POSTGRES_CONFIG["port"],
    dbname=POSTGRES_CONFIG["db"],
    user=POSTGRES_CONFIG["user"],
    password=POSTGRES_CONFIG["password"],
)

# List tables
cursor = conn.cursor()
cursor.execute("""
    SELECT table_name 
    FROM information_schema.tables 
    WHERE table_schema = 'public'
    ORDER BY table_name;
""")
print("=== Tables ===")
for row in cursor.fetchall():
    print(f"  {row[0]}")

# Query customers
cursor.execute("SELECT COUNT(*) FROM customers;")
print(f"\nCustomers: {cursor.fetchone()[0]} rows")

cursor.execute("SELECT COUNT(*) FROM claims;")
print(f"Claims: {cursor.fetchone()[0]} rows")

cursor.close()
conn.close()

## Query ClickHouse

In [ ]:
import clickhouse_connect

CLICKHOUSE_CONFIG = {
    "host": os.getenv("CLICKHOUSE_HOST", "localhost"),
    "port": os.getenv("CLICKHOUSE_PORT", "8123"),
    "database": os.getenv("CLICKHOUSE_DB", "insurance_db"),
    "username": os.getenv("CLICKHOUSE_USER", "default"),
    "password": os.getenv("CLICKHOUSE_PASSWORD", "clickhouse_pass"),
}

ch_client = clickhouse_connect.get_client(
    host=CLICKHOUSE_CONFIG["host"],
    port=CLICKHOUSE_CONFIG["port"],
    database=CLICKHOUSE_CONFIG["database"],
    username=CLICKHOUSE_CONFIG["username"],
    password=CLICKHOUSE_CONFIG["password"],
)

# List tables
result = ch_client.query("SHOW TABLES")
print("=== Tables ===")
for row in result.result_rows:
    print(f"  {row[0]}")